# 04 — ByteTrack + Zero-Shot Team Assignment (last 10 seconds)

This notebook is deliberately simpler than the previous hierarchical tracker.

It is inspired by the uploaded `basketball_analysis` project:

**RF-DETR detections → ByteTrack → zero-shot jersey classification → 5 persistent roster slots per team → annotated video**

For this SportsMOT sequence we process **only the last 10 seconds** to iterate quickly.

### Goal

Produce an annotated video with:

- max **5 persistent IDs for the blue team** (`B1...B5`)
- max **5 persistent IDs for the white team** (`W1...W5`)
- referees excluded
- ByteTrack used for short-term tracking
- a small slot-stabilization layer used to reduce visible ID changes

> This notebook does **not overwrite** the previous baseline.


In [ ]:
# If needed, install once:
# %pip install -U supervision transformers scipy pillow
# rfdetr must already be available in the environment used by Notebook 03.

import sys
import time
from pathlib import Path
from collections import defaultdict, deque

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

import supervision as sv
from transformers import CLIPProcessor, CLIPModel
from rfdetr import RFDETRNano

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("Supervision:", sv.__version__)
print("MPS available:", torch.backends.mps.is_available())


In [ ]:
# ============================================================
# PATHS / DATASET
# ============================================================

DRIVE_ROOT = Path(
    "/Users/marcodalbis/Library/CloudStorage/"
    "GoogleDrive-marco.dalbis@gmail.com/"
    "Il mio Drive/"
    "basketball-thesis"
)

SEQUENCE_NAME = "v_00HRwkvvjtQ_c001"

FRAMES_DIR = (
    DRIVE_ROOT
    / "datasets"
    / "sportsmot"
    / "val"
    / SEQUENCE_NAME
    / "img1"
)

CHECKPOINT_PATH = (
    DRIVE_ROOT
    / "checkpoints"
    / "rfdetr_nano_10epochs"
    / "checkpoint_best_total.pth"
)

OUTPUT_DIR = (
    DRIVE_ROOT
    / "experiments"
    / "bytetrack_clip_last10s"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VIDEO_FPS = 25
SECONDS_TO_PROCESS = 10

frame_paths = sorted(FRAMES_DIR.glob("*.jpg"))
NUM_FRAMES = len(frame_paths)

N_LAST_FRAMES = min(
    NUM_FRAMES,
    VIDEO_FPS * SECONDS_TO_PROCESS
)

START_FRAME = NUM_FRAMES - N_LAST_FRAMES + 1
END_FRAME = NUM_FRAMES

print("Sequence:", SEQUENCE_NAME)
print("Total frames:", NUM_FRAMES)
print("Processing frames:", START_FRAME, "->", END_FRAME)
print("Frames processed:", N_LAST_FRAMES)
print("Approx duration:", N_LAST_FRAMES / VIDEO_FPS, "s")
print("Output:", OUTPUT_DIR)

assert NUM_FRAMES > 0
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DETECTION_THRESHOLD = 0.30
PLAYER_CLASS_ID = 0

# Zero-shot labels for THIS sequence.
# For another match, only these descriptions need to be changed.
TEAM_PROMPTS = [
    "a basketball player wearing a blue jersey",
    "a basketball player wearing a white jersey",
    "a basketball referee wearing a black and white striped shirt",
]

BLUE_LABEL = 0
WHITE_LABEL = 1
REFEREE_LABEL = 2

# We re-evaluate jersey appearance periodically and use temporal voting.
TEAM_RECLASSIFY_EVERY = 12
TEAM_VOTE_HISTORY = 5
MIN_TEAM_PROBABILITY = 0.45

# Slot stabilizer: exactly five possible identities per team.
MAX_SLOTS_PER_TEAM = 5
SLOT_MAX_MISSED = 20

# Cost weights for slot assignment.
SLOT_IOU_WEIGHT = 0.55
SLOT_CENTER_WEIGHT = 0.35
SLOT_TRACKER_STICKY_WEIGHT = 0.10

# Reject extremely implausible matches.
MAX_NORMALIZED_CENTER_DISTANCE = 0.35

print("Team prompts:")
for i, p in enumerate(TEAM_PROMPTS):
    print(i, p)


In [ ]:
# ============================================================
# LOAD RF-DETR
# ============================================================

detector = RFDETRNano(
    pretrain_weights=str(CHECKPOINT_PATH)
)

print("RF-DETR loaded.")


In [ ]:
# ============================================================
# LOAD FASHION-CLIP
# ============================================================

CLIP_MODEL_NAME = "patrickjohncyh/fashion-clip"

clip_model = CLIPModel.from_pretrained(
    CLIP_MODEL_NAME
)

clip_processor = CLIPProcessor.from_pretrained(
    CLIP_MODEL_NAME
)

if torch.backends.mps.is_available():
    CLIP_DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    CLIP_DEVICE = torch.device("cuda")
else:
    CLIP_DEVICE = torch.device("cpu")

clip_model = clip_model.to(CLIP_DEVICE)
clip_model.eval()

print("Fashion-CLIP loaded on:", CLIP_DEVICE)


In [ ]:
# ============================================================
# RF-DETR -> SUPERVISION DETECTIONS
# ============================================================

def detect_players(frame_bgr):
    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    result = detector.predict(
        frame_rgb,
        threshold=DETECTION_THRESHOLD
    )

    xyxy = np.asarray(
        result.xyxy,
        dtype=np.float32
    )

    confidence = np.asarray(
        result.confidence,
        dtype=np.float32
    )

    class_id = np.asarray(
        result.class_id,
        dtype=int
    )

    if len(xyxy) == 0:
        return sv.Detections.empty()

    mask = (
        class_id == PLAYER_CLASS_ID
    )

    xyxy = xyxy[mask]
    confidence = confidence[mask]
    class_id = class_id[mask]

    if len(xyxy) == 0:
        return sv.Detections.empty()

    return sv.Detections(
        xyxy=xyxy,
        confidence=confidence,
        class_id=class_id
    )


In [ ]:
# ============================================================
# ZERO-SHOT TEAM / REFEREE CLASSIFICATION
# ============================================================

def safe_player_crop(frame_bgr, bbox):
    h, w = frame_bgr.shape[:2]

    x1, y1, x2, y2 = map(int, bbox)

    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h, y2))

    if x2 <= x1 or y2 <= y1:
        return None

    # Prefer torso over the full person crop.
    bw = x2 - x1
    bh = y2 - y1

    tx1 = int(x1 + 0.10 * bw)
    tx2 = int(x1 + 0.90 * bw)
    ty1 = int(y1 + 0.05 * bh)
    ty2 = int(y1 + 0.65 * bh)

    crop = frame_bgr[
        ty1:ty2,
        tx1:tx2
    ]

    if crop.size == 0:
        return None

    return crop


@torch.no_grad()
def classify_crops_zero_shot(crops_bgr):
    if len(crops_bgr) == 0:
        return []

    pil_images = []

    for crop in crops_bgr:
        rgb = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2RGB
        )
        pil_images.append(
            Image.fromarray(rgb)
        )

    inputs = clip_processor(
        text=TEAM_PROMPTS,
        images=pil_images,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        k: v.to(CLIP_DEVICE)
        for k, v in inputs.items()
    }

    outputs = clip_model(**inputs)

    probs = (
        outputs.logits_per_image
        .softmax(dim=1)
        .detach()
        .cpu()
        .numpy()
    )

    return probs


In [ ]:
# ============================================================
# GEOMETRY HELPERS
# ============================================================

def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter

    if union <= 1e-12:
        return 0.0

    return inter / union


def bbox_center(b):
    x1, y1, x2, y2 = map(float, b)
    return np.array(
        [
            (x1 + x2) / 2.0,
            (y1 + y2) / 2.0
        ],
        dtype=np.float32
    )


In [ ]:
# ============================================================
# 5-SLOT TEAM STABILIZER
# ============================================================

class TeamSlot:
    def __init__(self, slot_name):
        self.slot_name = slot_name
        self.last_bbox = None
        self.last_tracker_id = None
        self.last_seen_frame = -10**9
        self.missed = 10**9

    @property
    def initialized(self):
        return self.last_bbox is not None


class FiveSlotStabilizer:
    def __init__(self, prefix):
        self.prefix = prefix
        self.slots = [
            TeamSlot(f"{prefix}{i}")
            for i in range(1, MAX_SLOTS_PER_TEAM + 1)
        ]

    def _pair_cost(
        self,
        slot,
        detection,
        frame_diag
    ):
        bbox = detection["bbox"]
        tracker_id = detection["tracker_id"]

        if not slot.initialized:
            # Empty slots are allowed, but existing identities are preferred.
            return 0.55

        iou = bbox_iou(
            slot.last_bbox,
            bbox
        )

        c1 = bbox_center(
            slot.last_bbox
        )

        c2 = bbox_center(
            bbox
        )

        center_dist = float(
            np.linalg.norm(
                c1 - c2
            )
            / frame_diag
        )

        if center_dist > MAX_NORMALIZED_CENTER_DISTANCE:
            return 1e3

        sticky_penalty = (
            0.0
            if slot.last_tracker_id == tracker_id
            else 1.0
        )

        cost = (
            SLOT_IOU_WEIGHT
            * (1.0 - iou)
            +
            SLOT_CENTER_WEIGHT
            * center_dist
            +
            SLOT_TRACKER_STICKY_WEIGHT
            * sticky_penalty
        )

        # Slightly relax after a brief disappearance.
        if slot.missed > 0:
            cost += min(
                0.15,
                slot.missed * 0.01
            )

        return float(cost)

    def update(
        self,
        detections,
        frame_number,
        frame_diag
    ):
        # Increase missed count for every slot first.
        for slot in self.slots:
            slot.missed += 1

        if len(detections) == 0:
            return []

        cost = np.zeros(
            (
                len(self.slots),
                len(detections)
            ),
            dtype=np.float32
        )

        for i, slot in enumerate(self.slots):
            for j, det in enumerate(detections):
                cost[i, j] = self._pair_cost(
                    slot,
                    det,
                    frame_diag
                )

        rows, cols = linear_sum_assignment(
            cost
        )

        assigned = []

        for r, c in zip(rows, cols):
            if cost[r, c] >= 1e2:
                continue

            slot = self.slots[r]
            det = detections[c]

            slot.last_bbox = np.asarray(
                det["bbox"],
                dtype=np.float32
            )

            slot.last_tracker_id = (
                det["tracker_id"]
            )

            slot.last_seen_frame = (
                frame_number
            )

            slot.missed = 0

            assigned.append({
                **det,
                "slot_id": slot.slot_name
            })

        return assigned


## Main run

The next cell processes only the final 10 seconds.

The logic is:

1. RF-DETR detects persons/players.
2. ByteTrack gives short-term track IDs.
3. Fashion-CLIP votes on `blue / white / referee`.
4. Referees are discarded.
5. Each team is restricted to five persistent slots.
6. The output labels are `B1...B5` and `W1...W5`.

The original ByteTrack ID is also stored in the CSV for debugging.


In [ ]:
# ============================================================
# RUN: RF-DETR + BYTETRACK + CLIP + 5+5 SLOTS
# ============================================================

byte_tracker = sv.ByteTrack()

blue_slots = FiveSlotStabilizer("B")
white_slots = FiveSlotStabilizer("W")

track_team_votes = defaultdict(
    lambda: deque(
        maxlen=TEAM_VOTE_HISTORY
    )
)

track_last_classified = defaultdict(
    lambda: -10**9
)

rows_output = []

sample_frame = cv2.imread(
    str(frame_paths[START_FRAME - 1])
)

FRAME_H, FRAME_W = (
    sample_frame.shape[:2]
)

FRAME_DIAG = float(
    np.hypot(
        FRAME_W,
        FRAME_H
    )
)

start_time = time.perf_counter()


for frame_number in tqdm(
    range(
        START_FRAME,
        END_FRAME + 1
    ),
    desc="Tracking last 10 s"
):
    frame = cv2.imread(
        str(
            frame_paths[
                frame_number - 1
            ]
        )
    )

    if frame is None:
        continue

    detections = detect_players(
        frame
    )

    tracked = (
        byte_tracker
        .update_with_detections(
            detections
        )
    )

    current = []

    # supervision stores arrays on the Detections object.
    if len(tracked) > 0:
        for i in range(len(tracked)):
            tracker_id = int(
                tracked.tracker_id[i]
            )

            bbox = tracked.xyxy[i].astype(
                np.float32
            )

            conf = (
                float(tracked.confidence[i])
                if tracked.confidence is not None
                else 1.0
            )

            current.append({
                "tracker_id": tracker_id,
                "bbox": bbox,
                "confidence": conf,
                "team": None,
                "team_probability": None
            })

    # Decide which tracks need a fresh team classification.
    to_classify = []
    classify_indices = []

    for idx, det in enumerate(current):
        tid = det["tracker_id"]

        needs_update = (
            len(track_team_votes[tid]) == 0
            or
            frame_number - track_last_classified[tid]
            >= TEAM_RECLASSIFY_EVERY
        )

        if not needs_update:
            continue

        crop = safe_player_crop(
            frame,
            det["bbox"]
        )

        if crop is None:
            continue

        to_classify.append(crop)
        classify_indices.append(idx)

    if len(to_classify) > 0:
        probs_batch = (
            classify_crops_zero_shot(
                to_classify
            )
        )

        for idx, probs in zip(
            classify_indices,
            probs_batch
        ):
            det = current[idx]
            tid = det["tracker_id"]

            label = int(
                np.argmax(probs)
            )

            probability = float(
                probs[label]
            )

            if probability >= MIN_TEAM_PROBABILITY:
                track_team_votes[tid].append(
                    label
                )

            track_last_classified[tid] = (
                frame_number
            )

    # Convert votes into a stable semantic label.
    for det in current:
        tid = det["tracker_id"]
        votes = list(
            track_team_votes[tid]
        )

        if len(votes) == 0:
            continue

        counts = np.bincount(
            votes,
            minlength=len(TEAM_PROMPTS)
        )

        semantic = int(
            counts.argmax()
        )

        det["team"] = semantic
        det["team_probability"] = (
            counts[semantic]
            / len(votes)
        )

    blue_dets = [
        d for d in current
        if d["team"] == BLUE_LABEL
    ]

    white_dets = [
        d for d in current
        if d["team"] == WHITE_LABEL
    ]

    # Referee and unknown tracks are not sent to roster slots.
    blue_assigned = blue_slots.update(
        blue_dets,
        frame_number,
        FRAME_DIAG
    )

    white_assigned = white_slots.update(
        white_dets,
        frame_number,
        FRAME_DIAG
    )

    for team_name, assigned in [
        ("BLUE", blue_assigned),
        ("WHITE", white_assigned)
    ]:
        for item in assigned:
            x1, y1, x2, y2 = map(
                float,
                item["bbox"]
            )

            rows_output.append({
                "frame":
                    int(frame_number),

                "slot_id":
                    item["slot_id"],

                "team":
                    team_name,

                "bytetrack_id":
                    int(
                        item["tracker_id"]
                    ),

                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,

                "confidence":
                    float(
                        item["confidence"]
                    ),

                "team_vote_confidence":
                    float(
                        item[
                            "team_probability"
                        ]
                    )
            })


elapsed = (
    time.perf_counter()
    - start_time
)

tracking_10s_df = pd.DataFrame(
    rows_output
)

print("Done.")
print(f"Runtime: {elapsed:.1f} s")
print("Rows:", len(tracking_10s_df))
print(
    "Persistent output IDs:",
    sorted(
        tracking_10s_df[
            "slot_id"
        ].unique()
    )
)


In [ ]:
# ============================================================
# SANITY CHECK
# ============================================================

presence = (
    tracking_10s_df
    .groupby(
        [
            "frame",
            "team"
        ]
    )[
        "slot_id"
    ]
    .nunique()
    .unstack(
        fill_value=0
    )
)

for team in [
    "BLUE",
    "WHITE"
]:
    if team not in presence.columns:
        presence[team] = 0

print(
    "Max BLUE IDs/frame:",
    presence["BLUE"].max()
)

print(
    "Max WHITE IDs/frame:",
    presence["WHITE"].max()
)

print(
    "Unique BLUE slots:",
    sorted(
        tracking_10s_df[
            tracking_10s_df[
                "team"
            ] == "BLUE"
        ]["slot_id"].unique()
    )
)

print(
    "Unique WHITE slots:",
    sorted(
        tracking_10s_df[
            tracking_10s_df[
                "team"
            ] == "WHITE"
        ]["slot_id"].unique()
    )
)

display(
    presence.describe()
)


In [ ]:
# ============================================================
# OPTIONAL DEBUG: BYTE TRACK ID CHANGES INSIDE EACH SLOT
# ============================================================

slot_debug = (
    tracking_10s_df
    .groupby(
        [
            "team",
            "slot_id"
        ]
    )
    .agg(
        frames=("frame", "nunique"),
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
        raw_bytetrack_ids=(
            "bytetrack_id",
            lambda x: sorted(
                set(
                    map(int, x)
                )
            )
        )
    )
    .reset_index()
)

display(slot_debug)


In [ ]:
# ============================================================
# SAVE CSV
# ============================================================

CSV_PATH = (
    OUTPUT_DIR
    / f"{SEQUENCE_NAME}_last10s_bytetrack_clip_5x5.csv"
)

tracking_10s_df.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved:")
print(CSV_PATH)


In [ ]:
# ============================================================
# DRAW OUTPUT VIDEO
# ============================================================

VIDEO_PATH = (
    OUTPUT_DIR
    / f"{SEQUENCE_NAME}_last10s_bytetrack_clip_5x5.mp4"
)


def draw_frame(
    frame,
    rows
):
    output = frame.copy()

    for _, row in rows.iterrows():
        x1 = int(row["x1"])
        y1 = int(row["y1"])
        x2 = int(row["x2"])
        y2 = int(row["y2"])

        slot_id = str(
            row["slot_id"]
        )

        team = str(
            row["team"]
        )

        # One fixed annotation color is intentional:
        # identity is encoded by the text, not color.
        cv2.rectangle(
            output,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            output,
            f"{slot_id} | {team}",
            (
                x1,
                max(
                    22,
                    y1 - 7
                )
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    return output


first_frame = cv2.imread(
    str(
        frame_paths[
            START_FRAME - 1
        ]
    )
)

height, width = (
    first_frame.shape[:2]
)

writer = cv2.VideoWriter(
    str(VIDEO_PATH),
    cv2.VideoWriter_fourcc(
        *"mp4v"
    ),
    VIDEO_FPS,
    (
        width,
        height
    )
)


for frame_number in tqdm(
    range(
        START_FRAME,
        END_FRAME + 1
    ),
    desc="Writing video"
):
    frame = cv2.imread(
        str(
            frame_paths[
                frame_number - 1
            ]
        )
    )

    if frame is None:
        continue

    rows = tracking_10s_df[
        tracking_10s_df[
            "frame"
        ] == frame_number
    ]

    writer.write(
        draw_frame(
            frame,
            rows
        )
    )


writer.release()

print("Video saved:")
print(VIDEO_PATH)


In [ ]:
# ============================================================
# PREVIEW ONE FRAME
# ============================================================

preview_frame_number = (
    START_FRAME
    +
    (END_FRAME - START_FRAME) // 2
)

preview_frame = cv2.imread(
    str(
        frame_paths[
            preview_frame_number - 1
        ]
    )
)

preview_rows = tracking_10s_df[
    tracking_10s_df["frame"]
    == preview_frame_number
]

preview = draw_frame(
    preview_frame,
    preview_rows
)

plt.figure(
    figsize=(16, 9)
)

plt.imshow(
    cv2.cvtColor(
        preview,
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    f"Frame {preview_frame_number}"
)

plt.axis("off")
plt.show()


## What to inspect in the output

Do **not** judge this notebook only from the number of IDs. Watch the 10-second video and check:

1. Does each blue player keep the same `B1...B5` label through contacts?
2. Does each white player keep the same `W1...W5` label?
3. Are referees excluded?
4. When ByteTrack changes its raw ID, does the persistent slot stay stable?
5. Are there any obvious slot swaps during crossings?

If the video is visually good, this becomes a clean simplified tracking baseline inspired by the online project. If one or two slots swap, tune only the slot-assignment cost rather than rebuilding the entire pipeline.
